In [10]:
import pandas as pd
import time

from get_entities_ft_nlp import extract_ft_entities
from get_entities_LLM import extract_llm_entities
from get_sentiment_llm import extract_llm_sentiment
from get_sentiment_nlp import extract_nlp_sentiment

In [6]:
data = pd.read_csv("holdout_eval_set.csv")

In [7]:
input = list(data["Sentence"])

In [13]:
results = {}

# Define all combinations
combinations = [
    ("FT_entities", "NLP_sentiment", extract_ft_entities, extract_nlp_sentiment),
    ("FT_entities", "LLM_sentiment", extract_ft_entities, extract_llm_sentiment),
    ("LLM_entities", "NLP_sentiment", extract_llm_entities, extract_nlp_sentiment),
    ("LLM_entities", "LLM_sentiment", extract_llm_entities, extract_llm_sentiment),
]

for ent_name, sent_name, ent_func, sent_func in combinations:
    # Entity extraction timing
    start_ent = time.time()
    entities = ent_func(input)
    end_ent = time.time()
    ent_time = end_ent - start_ent

    # Prepare input for sentiment extraction to match main app logic
    if sent_func is extract_nlp_sentiment:
        # NLP sentiment expects: List[Tuple[str, List[str]]]
        nlp_input = []
        for sent, ents in zip(input, entities):
            # ents may be list of dicts or list of strings
            if isinstance(ents, list) and ents and isinstance(ents[0], dict) and "name" in ents[0]:
                ent_names = [e["name"] for e in ents]
            else:
                ent_names = ents
            nlp_input.append((sent, ent_names))
        entities_for_sentiment = nlp_input
    else:
        # LLM sentiment expects: List[{"sentence": ..., "entities": [{"name": ...}, ...]}]
        pipeline = []
        for sent, ents in zip(input, entities):
            ents_list = []
            for e in ents:
                if isinstance(e, dict) and "name" in e:
                    ents_list.append(e)
                else:
                    ents_list.append({"name": e})
            pipeline.append({"sentence": sent, "entities": ents_list})
        entities_for_sentiment = pipeline

    # Sentiment extraction timing
    start_sent = time.time()
    sentiments = sent_func(entities_for_sentiment)
    end_sent = time.time()
    sent_time = end_sent - start_sent

    # Store results
    results[(ent_name, sent_name)] = {
        "entity_time_sec": ent_time,
        "sentiment_time_sec": sent_time,
        "total_time_sec": ent_time + sent_time,
    }

# Print results in a readable format
print("Runtime comparison (seconds):")
for (ent_name, sent_name), times in results.items():
    print(f"Entities: {ent_name:12s} | Sentiment: {sent_name:12s} | "
          f"Entity Time: {times['entity_time_sec']:.2f} | "
          f"Sentiment Time: {times['sentiment_time_sec']:.2f} | "
          f"Total: {times['total_time_sec']:.2f}")


2025-08-27 12:17:30,106 | INFO | [extract] received items=400 | with_entities=400 | skipped=0
2025-08-27 12:17:30,106 | INFO | [extract] batching | chunks=20 | chunk_size≈20
2025-08-27 12:17:30,650 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:30,651 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:30,652 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:30,782 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:44,986 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:45,395 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:47,239 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-27 12:17:51,065 | INFO | HTTP Requ

Runtime comparison (seconds):
Entities: FT_entities  | Sentiment: NLP_sentiment | Entity Time: 10.04 | Sentiment Time: 11.04 | Total: 21.08
Entities: FT_entities  | Sentiment: LLM_sentiment | Entity Time: 9.71 | Sentiment Time: 86.37 | Total: 96.08
Entities: LLM_entities | Sentiment: NLP_sentiment | Entity Time: 119.59 | Sentiment Time: 11.15 | Total: 130.75
Entities: LLM_entities | Sentiment: LLM_sentiment | Entity Time: 118.79 | Sentiment Time: 88.38 | Total: 207.17


In [14]:
avg_obs = data.groupby("Document_ID").size().mean()
print(f"Average observation count per Document_ID: {avg_obs:.2f}")

Average observation count per Document_ID: 33.33
